## Import Libraries

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Random Forest Classifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier 

## XGBoost Classifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb

## Multinomial Logistic Regresion
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Support Vector Classifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

# SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline


## Import Dataset

In [ ]:
focus_df = pd.read_csv('Dataset/cleaned/focus_df.csv', parse_dates=['Date'])

## About Dataset

In [ ]:
focus_df.head(3)

In [ ]:
focus_df.shape

In [ ]:
focus_df.info()

In [ ]:
focus_df.isna().sum()

In [ ]:
focus_df.describe()

In [ ]:
train_df = focus_df[(focus_df['Date'] < '2025-01-01')]
test_df = focus_df[(focus_df['Date'] >= '2025-01-01')]

In [ ]:
train_df.shape, test_df.shape

#### Grouping and aggregating columns

In [ ]:
def aggregate(df):
    agg_df = df.groupby(['Barangay', 'Month', 'Weekday', 'Time_of_Day']).agg(
        Crime_Count=('Offense ID', 'count'),
        
        # Temporal features (take mode or first for consistency)
        Avg_Hour = ('Hour', 'mean'),
        Mode_Hour=('Hour', lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0]),
        Weekend_Crimes=('Is_Weekend', 'sum'),
        Weekday_Crimes=('Is_Weekend', lambda x: (~x.astype(bool)).sum()),
        
        # Spatial/demographic (constant per barangay, use first)
        Population=('Population_2024', 'first'),
        Pop_Density=('Pop_Density_2024', 'first'),
        Area_sqkm=('Area_sqkm', 'first'),
        
        # Police presence
        Avg_Distance_Police=('Distance_from_Police', 'mean'),
        Avg_Num_Stations_1km=('Num_Police_Stations_1km', 'mean'),
        
        # Crime characteristics
        Avg_Victims=('Num_Victims', 'median'),
        Avg_Suspects=('Num_Suspects', 'median'),
        
        # Focus crime distribution
        Murder_Count=('Focus_Crime', lambda x: (x == 'Murder').sum()),
        Theft_Count=('Focus_Crime', lambda x: (x == 'Theft').sum()),
        Robbery_Count=('Focus_Crime', lambda x: (x == 'Robbery').sum()),
        Physical_Injuries_Count=('Focus_Crime', lambda x: (x == 'Physical Injuries').sum()),
        Rape_Count=('Focus_Crime', lambda x: (x == 'Rape').sum()),
        Homicide_Count=('Focus_Crime', lambda x: (x == 'Homicide').sum()),
        Carnapping_MC_Count=('Focus_Crime', lambda x: (x == 'Carnapping MC').sum()),
        Carnapping_MV_Count=('Focus_Crime', lambda x: (x == 'Carnapping MV').sum())
    ).reset_index()

    agg_df['Crime_Rate_per_1000'] = (agg_df['Crime_Count'] / agg_df['Population']) * 1000
    agg_df['Crime_Density_sqkm'] = agg_df['Crime_Count'] / agg_df['Area_sqkm']
    agg_df['Weekend_Ratio'] = agg_df['Weekend_Crimes'] / (agg_df['Crime_Count'] + 1e-6)

    agg_df = agg_df.sort_values(['Barangay', 'Month', 'Weekday', 'Time_of_Day'])

    print(f"Aggregated dataset shape: {agg_df.shape}")

    return agg_df

In [ ]:
train_df = aggregate(train_df)
test_df = aggregate(test_df)

In [ ]:
train_df.head(2)

#### Applying Alarm Level
Applying alarm level on new dataset (new_df) based on thresholds from trained dataset (train_df)

In [ ]:
def classify_alarm(new_df, train_df):

    q25_baseline = train_df['Crime_Count'].quantile(0.25)
    q75_baseline = train_df['Crime_Count'].quantile(0.75)

    print(f"BASELINE THRESHOLDS (from 2017-2024 data):")
    print(f"   25th Percentile (Low/Medium boundary): {q25_baseline}")
    print(f"   75th Percentile (Medium/High boundary): {q75_baseline}")
    print(f" Mean of Crime Count: {train_df['Crime_Count'].mean().round(2)}")

    # Define a UNIVERSAL classification function using baseline thresholds
    def classify_alarm(count, q25, q75):
        if count <= q25:
            return 'Low'
        elif count <= q75:
            return 'Medium'
        else:
            return 'High'
    
    # Apply to 2017-2024 data
    train_df['Alarm_Level'] = train_df['Crime_Count'].apply(
    lambda x: classify_alarm(x, q25_baseline, q75_baseline)
)

    # Apply to 2025 data using THE SAME THRESHOLDS
    new_df['Alarm_Level'] = new_df['Crime_Count'].apply(
    lambda x: classify_alarm(x, q25_baseline, q75_baseline)
)
    # Verify distributions
    print("\n2017-2024 Alarm Level Distribution:")
    print(train_df['Alarm_Level'].value_counts().sort_index())

    print("\n2025 Alarm Level Distribution:")
    print(new_df['Alarm_Level'].value_counts().sort_index())

    print("\nNote: Both datasets now use the same crime alarm thresholds from 2017-2024 baseline!")
  

In [ ]:
classify_alarm(test_df, train_df)

## Correlation

#### Correlation of enriched_df

In [ ]:
enriched_df = pd.read_csv('Dataset/cleaned/enriched_df.csv', parse_dates=['Date'])
enriched_df = aggregate(enriched_df)

enr_cols = [
    'Crime_Count',  # Temporal
    'Avg_Hour', 'Weekend_Ratio',  # Time patterns
    'Population', 'Mode_Hour', 'Pop_Density', 'Area_sqkm',  # Demographic/spatial
     'Avg_Num_Stations_1km',  # Police presence
    'Avg_Victims', 'Avg_Suspects',  # Crime characteristics
    'Crime_Rate_per_1000', 'Crime_Density_sqkm',  # Derived metrics
]

corr_matrix_enr = enriched_df[enr_cols].corr(numeric_only=True)

plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix_enr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix - Enriched Dataset')
plt.show()


crime_corr_enr = corr_matrix_enr['Crime_Count'].sort_values(ascending=False)
print("Top correlations with Crime_Count (Enriched Dataset):")
print(crime_corr_enr)


#### Correlation of focus_df (2017-2025)

In [ ]:
focus_df_agg = aggregate(focus_df)

cols = [
    'Crime_Count',   # Temporal
    'Avg_Hour', 'Weekend_Ratio',  # Time patterns
    'Population', 'Mode_Hour', 'Pop_Density', 'Area_sqkm',  # Demographic/spatial
     'Avg_Num_Stations_1km',  # Police presence
    'Avg_Victims', 'Avg_Suspects',  # Crime characteristics
    'Crime_Rate_per_1000', 'Crime_Density_sqkm',  # Derived metrics
]
corr_matrix = focus_df_agg[cols].corr(numeric_only=True)

plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of 2017-2024 Dataset Features', fontsize=16)
plt.show()

# Focus on Crime_Count correlations
crime_corr = corr_matrix['Crime_Count'].sort_values(ascending=False)
print("\nTop Predictors for Crime_Count:\n", crime_corr)

#### Correlation of train_df (2017-2024)

In [ ]:
cols = [
    'Crime_Count',   # Temporal
    'Avg_Hour', 'Weekend_Ratio',  # Time patterns
    'Population', 'Mode_Hour', 'Pop_Density', 'Area_sqkm',  # Demographic/spatial
     'Avg_Num_Stations_1km',  # Police presence
    'Avg_Victims', 'Avg_Suspects',  # Crime characteristics
    'Crime_Rate_per_1000', 'Crime_Density_sqkm',  # Derived metrics
]

print(train_df.shape)

corr_matrix = train_df[cols].corr(numeric_only=True)

plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of 2017-2024 Dataset Features', fontsize=16)
plt.show()

# Focus on Crime_Count correlations
crime_corr = corr_matrix['Crime_Count'].sort_values(ascending=False)
print("\nTop Predictors for Crime_Count:\n", crime_corr)

## Feature selection

In [ ]:
selected_features = [       # Two months ago
    'Population',              # Demographic
    'Area_sqkm',               # Spatial
    'Avg_Num_Stations_1km',    # Police presence
    'Weekend_Ratio',           # Temporal pattern
    'Avg_Hour',                # Time of day
    'Avg_Victims',             # Crime severity
    'Avg_Suspects'             # Crime characteristics
]

rand_seed = 42
cv = 5


# Features without Past Crime Data

## Random Forest

In [ ]:
rf_df = train_df.copy()

print("Shape:", rf_df.shape)
rf_df.head(3)

#### With class imbalance

In [ ]:
# ----------FEATURE SELECTION--------

RF_X = rf_df[selected_features]
RF_y = rf_df['Alarm_Level']

#Encoding target variable
RF_y = RF_y.map({'Low': 0, 'Medium': 1, 'High': 2})

# ----------TRAIN AND TEST SPLIT----------
test_size = [0.25]

for size in test_size:

    RF_X_train, RF_X_test, RF_y_train, RF_y_test = train_test_split(RF_X, RF_y, test_size=size, random_state=rand_seed, stratify=RF_y)

    print(f"\n--- Random Forest Classifier with {size*100}% Test Size ---")
    print("RF_X_train.shape:", RF_X_train.shape)
    print("RF_y_train.shape:", RF_y_train.shape)
    print("\nRF_X_test.shape:", RF_X_test.shape)
    print("RF_y_test.shape:", RF_y_test.shape)


# ----------PARAMETER SELECTION-------------

    RF_param_grid = {
    # Number of trees. More is generally better but has diminishing returns.
    'n_estimators': [100, 200, 300, 500],
    
    # Maximum depth of each tree. Crucial for preventing overfitting.
    'max_depth': [8, 10, 12, 15],
    
    # Minimum number of samples required to split a node.
    'min_samples_split': [5, 10, 15],
    
    # Minimum number of samples required at a leaf node.
    'min_samples_leaf': [4, 6, 8, 10, 12],
    
    # Number of features to consider when looking for the best split.
    'max_features': ['sqrt', 'log2'] 
    }   

    rf = RandomForestClassifier(random_state=rand_seed, class_weight='balanced')

    RF_grid_search = GridSearchCV(estimator=rf, param_grid=RF_param_grid, cv=cv, scoring='accuracy', verbose=0, n_jobs=-1)
    RF_grid_search.fit(RF_X_train, RF_y_train)

    rf_model = RandomForestClassifier(**RF_grid_search.best_params_)
    rf_model.fit(RF_X_train, RF_y_train)
    RF_y_pred = rf_model.predict(RF_X_test)

    # ----------RESULTS----------

    print(f"\n--- Random Forest Classifier with {size*100}% Test Size Results ---")
    print("Best Parameters:\n", RF_grid_search.best_params_)
    test_accuracy = accuracy_score(RF_y_test, RF_y_pred)

    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print('CV mean:', RF_grid_search.best_score_)
    print('The fit score:', rf_model.score(RF_X_train, RF_y_train))

    # Determine if gap is underfit, overfit, or good fit
    gap_score = rf_model.score(RF_X_train, RF_y_train) - test_accuracy

    if gap_score < 0.05:
        gap = 'Good Fit'
    elif gap_score >= 0.05 and gap_score < 0.15:
        gap = 'Overfit'
    else:
        gap = 'Underfit' 

    print('Underfit or Overfit?:', gap_score, gap)
    print("\nClassification Report:\n", classification_report(RF_y_test, RF_y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(RF_y_test, RF_y_pred))

#### Without class imbalance (SMOTE - 'Auto' oversampling technique)

In [ ]:
# ----------FEATURE SELECTION--------

RF_X = rf_df[selected_features]
RF_y = rf_df['Alarm_Level']

#Encoding target variable
RF_y = RF_y.map({'Low': 0, 'Medium': 1, 'High': 2})

# ----------TRAIN AND TEST SPLIT----------
test_size = [0.25]

for size in test_size:

    RF_X_train, RF_X_test, RF_y_train, RF_y_test = train_test_split(RF_X, RF_y, test_size=size, random_state=rand_seed, stratify=RF_y)

    print(f"\n--- Random Forest Classifier with {size*100}% Test Size ---")
    print("RF_X_train.shape:", RF_X_train.shape)
    print("RF_y_train.shape:", RF_y_train.shape)
    print("\nRF_X_test.shape:", RF_X_test.shape)
    print("RF_y_test.shape:", RF_y_test.shape)

# ----------SMOTE----------


    pipeline = ImbPipeline([('smote', SMOTE(sampling_strategy='auto', random_state=rand_seed))])

    RF_X_train, RF_y_train = pipeline.fit_resample(RF_X_train, RF_y_train)
    print("\nAfter SMOTE, RF_X_train.shape:", RF_X_train.shape)
    print("After SMOTE, RF_y_train.shape:", RF_y_train.shape, "\n")


 # ----------PARAMETER SELECTION-------------

    RF_param_grid = {
    # Number of trees. More is generally better but has diminishing returns.
    'n_estimators': [100, 200, 300, 500],
    
    # Maximum depth of each tree. Crucial for preventing overfitting.
    'max_depth': [4, 8, 10, 12, 15],
    
    # Minimum number of samples required to split a node.
    'min_samples_split': [3, 6, 10, 12],
    
    # Minimum number of samples required at a leaf node.
    'min_samples_leaf': [4, 6, 8, 10, 12],
    
    # Number of features to consider when looking for the best split.
    'max_features': ['sqrt', 'log2'] 
    }   

    rf = RandomForestClassifier(random_state=rand_seed, class_weight='balanced')

    RF_grid_search = GridSearchCV(estimator=rf, param_grid=RF_param_grid, cv=cv, scoring='accuracy', verbose=0, n_jobs=-1)
    RF_grid_search.fit(RF_X_train, RF_y_train)

    rf_model = RandomForestClassifier(**RF_grid_search.best_params_)
    rf_model.fit(RF_X_train, RF_y_train)
    RF_y_pred = rf_model.predict(RF_X_test)

    # ----------RESULTS----------

    print(f"\n--- Random Forest Classifier with {size*100}% Test Size Results ---")
    print("Best Parameters:\n", RF_grid_search.best_params_)
    test_accuracy = accuracy_score(RF_y_test, RF_y_pred)

    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print('CV mean:', RF_grid_search.best_score_)
    print('The fit score:', rf_model.score(RF_X_train, RF_y_train))

    # Determine if gap is underfit, overfit, or good fit
    gap_score = rf_model.score(RF_X_train, RF_y_train) - test_accuracy

    if gap_score < 0.05:
        gap = 'Good Fit'
    elif gap_score >= 0.05 and gap_score < 0.15:
        gap = 'Overfit'
    else:
        gap = 'Underfit' 

    print('Underfit or Overfit?:', gap_score, gap)
    print("\nClassification Report:\n", classification_report(RF_y_test, RF_y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(RF_y_test, RF_y_pred))

## XGBoost Classifier

In [ ]:
xgb_df = train_df.copy()

xgb_df.shape

#### With class imbalance

In [ ]:
selected_features = [       # Two months ago
    'Population',              # Demographic
    'Area_sqkm',               # Spatial
    'Avg_Num_Stations_1km',    # Police presence
    'Weekend_Ratio',           # Temporal pattern
    'Avg_Hour',                # Time of day
    'Avg_Victims',             # Crime severity
    'Avg_Suspects',
               # Crime characteristics
]

In [ ]:
# --------- FEATURE SELECTION--------

XGB_X = xgb_df[selected_features]
XGB_y = xgb_df['Alarm_Level']

XGB_y = XGB_y.map({'Low': 0, 'Medium': 1, 'High': 2})

# ------ TRAIN AND TEST SPLIT--------

size = 0.25

XGB_X_train, XGB_X_test, XGB_y_train, XGB_y_test = train_test_split(XGB_X, XGB_y, test_size=size, random_state=rand_seed, stratify=XGB_y)

# --------- MODEL TRAINING & PARAMETERS SELECTION -----------------

param_grid = {
# Controls the complexity of the trees
'max_depth': [3, 4, 5, 6, 7],
    
# Controls the step size. Smaller values require more trees.
 'learning_rate': [0.01, 0.05, 0.1],
    
# Number of boosting rounds.
'n_estimators': [100, 200, 300, 400],
    
# Regularization parameters to prevent overfitting
'gamma': [0, 0.1, 0.5], # Minimum loss reduction to make a split
'subsample': [0.7, 0.8, 0.9], # Fraction of training data to use per tree
'colsample_bytree': [0.7, 0.8, 0.9], # Fraction of features to use per tree
    
# L1 and L2 regularization
'reg_alpha': [0, 0.01, 0.1], # L1 regularization
'reg_lambda': [1, 1.5, 2] # L2 regularization
}

xgb_model = xgb.XGBClassifier(random_state=rand_seed)

xgb_grid_search = RandomizedSearchCV(xgb_model, param_grid, cv=cv, scoring="accuracy", n_jobs=-1, random_state=rand_seed)
xgb_grid_search.fit(XGB_X_train, XGB_y_train)

best_xgb = xgb_grid_search.best_estimator_

xgb_y_pred = best_xgb.predict(XGB_X_test)

# ----------------RESULTS------------------

print(f"\n--- XGBoost Classifier with {size*100}% Test Size Results ---")
print("Best Parameters:\n", xgb_grid_search.best_params_)
test_accuracy = accuracy_score(XGB_y_test, xgb_y_pred)

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print('CV mean:', xgb_grid_search.best_score_)
print('The fit score:', best_xgb.score(XGB_X_train, XGB_y_train))

# Determine if gap is underfit, overfit, or good fit
gap_score = best_xgb.score(XGB_X_train, XGB_y_train) - test_accuracy

if gap_score < 0.05:
    gap = 'Good Fit'
elif gap_score >= 0.05 and gap_score < 0.15:
    gap = 'Overfit'
else:
    gap = 'Underfit' 

print('Underfit or Overfit?:', gap_score, gap)
print("\nClassification Report:\n", classification_report(XGB_y_test, xgb_y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(XGB_y_test, xgb_y_pred))


In [ ]:
# --------- FEATURE SELECTION--------

XGB_X = xgb_df[selected_features]
XGB_y = xgb_df['Alarm_Level']

XGB_y = XGB_y.map({'Low': 0, 'Medium': 1, 'High': 2})

# ------ TRAIN AND TEST SPLIT--------

size = 0.25

XGB_X_train, XGB_X_test, XGB_y_train, XGB_y_test = train_test_split(XGB_X, XGB_y, test_size=size, random_state=rand_seed, stratify=XGB_y)

# --------- MODEL TRAINING & PARAMETERS SELECTION -----------------

param_grid = {
# Controls the complexity of the trees
'max_depth': [5],
    
# Controls the step size. Smaller values require more trees.
 'learning_rate': [0.05],
    
# Number of boosting rounds.
'n_estimators': [400],
    
# Regularization parameters to prevent overfitting
'gamma': [0], # Minimum loss reduction to make a split
'subsample': [0.9], # Fraction of training data to use per tree
'colsample_bytree': [0.7], # Fraction of features to use per tree
    
# L1 and L2 regularization
'reg_alpha': [0.01], # L1 regularization
'reg_lambda': [1.5] # L2 regularization
}

xgb_model = xgb.XGBClassifier(random_state=rand_seed)

xgb_grid_search = RandomizedSearchCV(xgb_model, param_grid, cv=cv, scoring="accuracy", n_jobs=-1, random_state=rand_seed)
xgb_grid_search.fit(XGB_X_train, XGB_y_train)

best_xgb = xgb_grid_search.best_estimator_

xgb_y_pred = best_xgb.predict(XGB_X_test)

# ----------------RESULTS------------------

print(f"\n--- XGBoost Classifier with {size*100}% Test Size Results ---")
print("Best Parameters:\n", xgb_grid_search.best_params_)
test_accuracy = accuracy_score(XGB_y_test, xgb_y_pred)

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print('CV mean:', xgb_grid_search.best_score_)
print('The fit score:', best_xgb.score(XGB_X_train, XGB_y_train))

# Determine if gap is underfit, overfit, or good fit
gap_score = best_xgb.score(XGB_X_train, XGB_y_train) - test_accuracy

if gap_score < 0.05:
    gap = 'Good Fit'
elif gap_score >= 0.05 and gap_score < 0.15:
    gap = 'Overfit'
else:
    gap = 'Underfit' 

print('Underfit or Overfit?:', gap_score, gap)
print("\nClassification Report:\n", classification_report(XGB_y_test, xgb_y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(XGB_y_test, xgb_y_pred))


#### Test prediction on 2025 data using trained 2017-2024 XGBoost Model

In [ ]:
X_2025 = test_df[selected_features]
y_2025 = test_df['Alarm_Level']

y_2025 = y_2025.map({'Low': 0, 'Medium': 1, 'High': 2})

y_2025_pred = best_xgb.predict(X_2025)

print("\n2025 Classification Report:\n", classification_report(y_2025, y_2025_pred))

## Print the 2025 data with predicted alarm levels
test_df_results = test_df.copy()
test_df_results['Predicted_Alarm_Level'] = y_2025_pred
print("\n2025 Barangay-Month Predictions:\n", test_df_results[['Barangay', 'Month', 'Time_of_Day', 'Crime_Count', 'Alarm_Level', 'Predicted_Alarm_Level']])

test_df_results[['Crime_Count', 'Alarm_Level']].value_counts()


#### Without class imbalance (SMOTE)

In [ ]:
# --------- FEATURE SELECTION--------

XGB_X = xgb_df[selected_features]
XGB_y = xgb_df['Alarm_Level']

XGB_y = XGB_y.map({'Low': 0, 'Medium': 1, 'High': 2})

# ------ TRAIN AND TEST SPLIT--------

size = 0.25

XGB_X_train, XGB_X_test, XGB_y_train, XGB_y_test = train_test_split(XGB_X, XGB_y, test_size=size, random_state=rand_seed, stratify=XGB_y)

# ----------SMOTE----------

pipeline = ImbPipeline([('smote', SMOTE(sampling_strategy='auto', random_state=rand_seed))])
XGB_X_train, XGB_y_train = pipeline.fit_resample(XGB_X_train, XGB_y_train)
print("\nAfter SMOTE, XGB_X_train.shape:", XGB_X_train.shape)
print("After SMOTE, XGB_y_train.shape:", XGB_y_train.shape, "\n")

# --------- MODEL TRAINING & PARAMETERS SELECTION -----------------

param_grid = {
# Controls the complexity of the trees
'max_depth': [3, 4, 5, 6, 7, 8, 10],
    
# Controls the step size. Smaller values require more trees.
 'learning_rate': [0.01, 0.05, 0.1, 0.03],
    
# Number of boosting rounds.
'n_estimators': [100, 200, 300, 400, 500, 1000],
    
# Regularization parameters to prevent overfitting
'gamma': [0, 0.1, 0.2, 0.5], # Minimum loss reduction to make a split
'subsample': [0.7, 0.8, 0.9], # Fraction of training data to use per tree
'colsample_bytree': [0.7, 0.8, 0.9], # Fraction of features to use per tree
    
# L1 and L2 regularization
'reg_alpha': [0, 0.01, 0.1], # L1 regularization
'reg_lambda': [1, 1.5, 2] # L2 regularization
}

xgb_model = xgb.XGBClassifier(random_state=rand_seed)

xgb_grid_search = RandomizedSearchCV(xgb_model, param_grid, cv=cv, scoring="accuracy", n_jobs=-1, random_state=rand_seed)
xgb_grid_search.fit(XGB_X_train, XGB_y_train)

best_xgb = xgb_grid_search.best_estimator_

xgb_y_pred = best_xgb.predict(XGB_X_test)

# ----------------RESULTS------------------

print(f"\n--- XGBoost Classifier with {size*100}% Test Size Results ---")
print("Best Parameters:\n", xgb_grid_search.best_params_)
test_accuracy = accuracy_score(XGB_y_test, xgb_y_pred)

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print('CV mean:', xgb_grid_search.best_score_)
print('The fit score:', best_xgb.score(XGB_X_train, XGB_y_train))

# Determine if gap is underfit, overfit, or good fit
gap_score = best_xgb.score(XGB_X_train, XGB_y_train) - test_accuracy

if gap_score < 0.05:
    gap = 'Good Fit'
elif gap_score >= 0.05 and gap_score < 0.15:
    gap = 'Overfit'
else:
    gap = 'Underfit' 

print('Underfit or Overfit?:', gap_score, gap)
print("\nClassification Report:\n", classification_report(XGB_y_test, xgb_y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(XGB_y_test, xgb_y_pred))


## Support Vector Classifier

In [ ]:
SVC_df = train_df.copy()

SVC_df.head(3)

#### With class imbalance

In [ ]:
# ---------- FEATURE SELECTION--------

SVC_X = SVC_df[selected_features]
SVC_y = SVC_df['Alarm_Level']

SVC_y = SVC_y.map({'Low': 0, 'Medium': 1, 'High': 2})


# ----------TRAIN AND TEST SPLIT----------

test_size = [0.25]

for size in test_size:

    SVC_X_train, SVC_X_test, SVC_y_train, SVC_y_test = train_test_split(SVC_X, SVC_y, test_size=size, random_state=rand_seed, stratify=SVC_y)
    print('\nSVC_X_train.shape:', SVC_X_train.shape)
    print('SVC_y_train.shape:', SVC_y_train.shape)
    print('\nSVC_X_test.shape:', SVC_X_test.shape)
    print('SVC_y_test.shape:', SVC_y_test.shape)

# -----FEATURE SCALING ---------------


    scaler = StandardScaler()
    SVC_X_train = scaler.fit_transform(SVC_X_train)
    SVC_X_test = scaler.transform(SVC_X_test)


# ---------- MODEL TRAINING & PARAMETERS SELECTION ----------

    param_grid = [
    {
        'kernel': ['rbf'], 
        'C': [1, 2, 5, 10, 15, 25, 50],
        'gamma': ['scale', 0.01, 0.02, 0.05, 0.1]
    },
    {
        'kernel': ['linear'], 
        'C': [1, 2, 5, 10, 15, 25, 50],
        'gamma': ['scale', 0.01, 0.02, 0.05, 0.1]
    }
]

    SVC_model = SVC( random_state=rand_seed)

    grid_search = GridSearchCV(estimator=SVC_model, param_grid=param_grid, cv=cv, scoring='accuracy', verbose=1, n_jobs=-1)
    grid_search.fit(SVC_X_train, SVC_y_train)

    best_SVC_model = SVC(**grid_search.best_params_, random_state=rand_seed)
    best_SVC_model.fit(SVC_X_train, SVC_y_train)

    SVC_y_pred = best_SVC_model.predict(SVC_X_test)

# -------------- RESULTS -------------
    
    print(f"\n--- SVC Classifier with {size*100}% Test Size Results ---")
    print("Best Parameters:\n", best_SVC_model.get_params())
    test_accuracy = accuracy_score(SVC_y_test, SVC_y_pred)

    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print('CV mean:', grid_search.best_score_)
    print('The fit score:', best_SVC_model.score(SVC_X_train, SVC_y_train))

    # Determine if gap is underfit, overfit, or good fit
    gap_score = best_SVC_model.score(SVC_X_train, SVC_y_train) - test_accuracy

    if gap_score < 0.05:
        gap = 'Good Fit'
    elif gap_score >= 0.05 and gap_score < 0.15:
        gap = 'Overfit'
    else:
        gap = 'Underfit' 

    print('Underfit or Overfit?:', gap_score, gap)
    print("\nClassification Report:\n", classification_report(SVC_y_test, SVC_y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(SVC_y_test, SVC_y_pred))



#### Without class imbalance (SMOTE - 'Auto' oversampling)

In [ ]:
# ---------- FEATURE SELECTION--------

SVC_X = SVC_df[selected_features]
SVC_y = SVC_df['Alarm_Level']

SVC_y = SVC_y.map({'Low': 0, 'Medium': 1, 'High': 2})


# ----------TRAIN AND TEST SPLIT----------

test_size = [0.25]

for size in test_size:

    SVC_X_train, SVC_X_test, SVC_y_train, SVC_y_test = train_test_split(SVC_X, SVC_y, test_size=size, random_state=rand_seed, stratify=SVC_y)
    print('\nSVC_X_train.shape:', SVC_X_train.shape)
    print('SVC_y_train.shape:', SVC_y_train.shape)
    print('\nSVC_X_test.shape:', SVC_X_test.shape)
    print('SVC_y_test.shape:', SVC_y_test.shape)

# ----------SMOTE----------


    pipeline = ImbPipeline([('smote', SMOTE(sampling_strategy='auto', random_state=rand_seed))])
    SVC_X_train, SVC_y_train = pipeline.fit_resample(SVC_X_train, SVC_y_train)
    print("\nAfter SMOTE, SVC_X_train.shape:", SVC_X_train.shape)
    print("After SMOTE, SVC_y_train.shape:", SVC_y_train.shape, "\n")


# -----FEATURE SCALING ---------------


    scaler = StandardScaler()
    SVC_X_train = scaler.fit_transform(SVC_X_train)
    SVC_X_test = scaler.transform(SVC_X_test)


# ---------- MODEL TRAINING & PARAMETERS SELECTION ----------

    param_grid = [
    {
        'kernel': ['rbf'], 
        'C': [1, 2, 5, 10, 15, 25, 50, 70, 100, 150],
        'gamma': ['scale', 0.01, 0.02, 0.05, 0.07, 0.1]
    },
    {
        'kernel': ['linear'], 
        'C': [1, 2, 5, 10, 15, 25, 50, 70, 100, 150],
        'gamma': ['scale', 0.01, 0.02, 0.05, 0.07, 0.1]
    }
]

    SVC_model = SVC(random_state=rand_seed)

    grid_search = GridSearchCV(estimator=SVC_model, param_grid=param_grid, cv=cv, scoring='accuracy', verbose=1, n_jobs=-1)
    grid_search.fit(SVC_X_train, SVC_y_train)

    best_SVC_model = SVC(**grid_search.best_params_, random_state=rand_seed)
    best_SVC_model.fit(SVC_X_train, SVC_y_train)

    SVC_y_pred = best_SVC_model.predict(SVC_X_test)

# -------------- RESULTS -------------
    
    print(f"\n--- SVC Classifier with {size*100}% Test Size Results ---")
    print("Best Parameters:\n", best_SVC_model.get_params())
    test_accuracy = accuracy_score(SVC_y_test, SVC_y_pred)

    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print('CV mean:', grid_search.best_score_)
    print('The fit score:', best_SVC_model.score(SVC_X_train, SVC_y_train))

    # Determine if gap is underfit, overfit, or good fit
    gap_score = best_SVC_model.score(SVC_X_train, SVC_y_train) - test_accuracy

    if gap_score < 0.05:
        gap = 'Good Fit'
    elif gap_score >= 0.05 and gap_score < 0.15:
        gap = 'Overfit'
    else:
        gap = 'Underfit' 

    print('Underfit or Overfit?:', gap_score, gap)
    print("\nClassification Report:\n", classification_report(SVC_y_test, SVC_y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(SVC_y_test, SVC_y_pred))



## Multinomial Logistic Regression

In [ ]:
MLR_df = train_df.copy()

#### With class imbalance

In [ ]:
# ----------FEATURE SELECTION--------

MLR_X = MLR_df[selected_features]
MLR_y = MLR_df['Alarm_Level']

MLR_y = MLR_y.map({'Low': 0, 'Medium': 1, 'High': 2})

# ----------TRAIN AND TEST SPLIT----------
test_size = [0.25]

for size in test_size:
    MLR_X_train, MLR_X_test, MLR_y_train, MLR_y_test = train_test_split(MLR_X, MLR_y, test_size=size, random_state=rand_seed, stratify=MLR_y)

    print('Shape of MLR_X_train:', MLR_X_train.shape)
    print('Shape of MLR_y_train:', MLR_y_train.shape)
    print('Shape of MLR_X_test:', MLR_X_test.shape)
    print('Shape of MLR_y_test:', MLR_y_test.shape, '\n')


    # -------FEATURE SCALING-------

    scaler = StandardScaler()
    MLR_X_train = scaler.fit_transform(MLR_X_train)
    MLR_X_test = scaler.transform(MLR_X_test)

    # ----------MODEL TRAINING----------

    param_grid = [
        {
            'solver': ['lbfgs'],
            'penalty': ['l2', ],
            'C': [0.01, 0.1, 1, 10],
        },
    ]

    MLR_model = LogisticRegression(multi_class='multinomial', class_weight='balanced', random_state=rand_seed)
    grid_search = GridSearchCV(MLR_model, param_grid, cv=cv)
    grid_search.fit(MLR_X_train, MLR_y_train)

    # ----------PARAMETER SELECTION------

    print("Best parameters found for MLR: ", grid_search.best_params_)
    MLR_model = LogisticRegression(**grid_search.best_params_, multi_class='multinomial', class_weight='balanced')

    # Fit scikit-learn LogisticRegression on the (already scaled) data
    MLR_model.fit(MLR_X_train, MLR_y_train)
    MLR_y_pred = MLR_model.predict(MLR_X_test)

    # ---------RESULTS-----------

    train_accuracy = MLR_model.score(MLR_X_train, MLR_y_train)
    test_accuracy = accuracy_score(MLR_y_test, MLR_y_pred)
    gap_score = train_accuracy - test_accuracy
    gap = ''
    if gap_score > 0.1:
        gap = 'Overfitting'
    elif gap_score < -0.1:
        gap = 'Underfitting'
    else:
        gap = 'Good Fit'

    print("\nMultinomial Logistic Regression", size, "test) Results:")
    print(f"MLR Training Accuracy: {train_accuracy:.4f}")
    print(f"MLR Test Accuracy: {test_accuracy:.4f}")
    print(f"MLR Gap Score: {gap_score:.4f} ({gap})")

    print('accuracy_score', accuracy_score(MLR_y_test, MLR_y_pred))
    print("Classification Report:\n", classification_report(MLR_y_test, MLR_y_pred, target_names=['Low', 'Medium', 'High']))
    test_accuracy = accuracy_score(MLR_y_test, MLR_y_pred)
    print(f"Test Accuracy: {test_accuracy:.4f}")



#### Without class imbalance

In [ ]:
# ----------FEATURE SELECTION--------

MLR_X = MLR_df[selected_features]
MLR_y = MLR_df['Alarm_Level']

MLR_y = MLR_y.map({'Low': 0, 'Medium': 1, 'High': 2})

# ----------TRAIN AND TEST SPLIT----------
test_size = [0.25]

for size in test_size:
    MLR_X_train, MLR_X_test, MLR_y_train, MLR_y_test = train_test_split(MLR_X, MLR_y, test_size=size, random_state=rand_seed, stratify=MLR_y)

    print('Shape of MLR_X_train:', MLR_X_train.shape)
    print('Shape of MLR_y_train:', MLR_y_train.shape)
    print('Shape of MLR_X_test:', MLR_X_test.shape)
    print('Shape of MLR_y_test:', MLR_y_test.shape, '\n')

    # ------SMOTE-----------

    pipeline = ImbPipeline([('smote', SMOTE(sampling_strategy='auto', random_state=rand_seed))])
    MLR_X_train, MLR_y_train = pipeline.fit_resample(MLR_X_train, MLR_y_train)
    print("After SMOTE, MLR_X_train.shape:", MLR_X_train.shape)
    print("After SMOTE, MLR_y_train.shape:", MLR_y_train.shape, "\n")

    # -------FEATURE SCALING-------

    scaler = StandardScaler()
    MLR_X_train = scaler.fit_transform(MLR_X_train)
    MLR_X_test = scaler.transform(MLR_X_test)

    # ----------MODEL TRAINING----------

    param_grid = [
        {
            'solver': ['lbfgs'],
            'penalty': ['l2', ],
            'C': [0.01, 0.1, 1, 10],
        },
    ]

    MLR_model = LogisticRegression(multi_class='multinomial', class_weight='balanced', random_state=rand_seed)
    grid_search = GridSearchCV(MLR_model, param_grid, cv=cv)
    grid_search.fit(MLR_X_train, MLR_y_train)

    # ----------PARAMETER SELECTION------

    print("Best parameters found for MLR: ", grid_search.best_params_)
    MLR_model = LogisticRegression(**grid_search.best_params_, multi_class='multinomial', class_weight='balanced')

    # Fit scikit-learn LogisticRegression on the (already scaled) data
    MLR_model.fit(MLR_X_train, MLR_y_train)
    MLR_y_pred = MLR_model.predict(MLR_X_test)

    # ---------RESULTS-----------

    train_accuracy = MLR_model.score(MLR_X_train, MLR_y_train)
    test_accuracy = accuracy_score(MLR_y_test, MLR_y_pred)
    gap_score = train_accuracy - test_accuracy
    gap = ''
    if gap_score > 0.1:
        gap = 'Overfitting'
    elif gap_score < -0.1:
        gap = 'Underfitting'
    else:
        gap = 'Good Fit'

    print("\nMultinomial Logistic Regression with SMOTE", size, "test) Results:")
    print(f"MLR Training Accuracy: {train_accuracy:.4f}")
    print(f"MLR Test Accuracy: {test_accuracy:.4f}")
    print(f"MLR Gap Score: {gap_score:.4f} ({gap})")

    print('accuracy_score', accuracy_score(MLR_y_test, MLR_y_pred))
    print("Classification Report:\n", classification_report(MLR_y_test, MLR_y_pred, target_names=['Low', 'Medium', 'High']))
    test_accuracy = accuracy_score(MLR_y_test, MLR_y_pred)
    print(f"Test Accuracy: {test_accuracy:.4f}")